# Warehouse Throughput Forecasting

## Project Overview

Forecasts **daily warehouse units shipped** over a 14-day horizon. Synthetic data spans 2 years with weekly operations patterns, seasonal retail peaks, and promotional surges.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily shipment volumes, predict the next 14 days. Warehouses need throughput forecasts for labor scheduling, dock allocation, and inventory staging.

## Why This Project Matters

Warehouse labor is the largest operating cost. Accurate throughput forecasts enable optimal shift scheduling, reduce overtime costs, prevent bottlenecks, and ensure on-time delivery SLAs are met.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily units shipped
- Weekly pattern (Mon-Fri operations, reduced weekends)
- Seasonal peaks (Black Friday, Christmas, back-to-school)
- Promotional sale surges
- Growth trend (e-commerce expansion)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `units_shipped` | Daily units shipped from warehouse |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'units_shipped'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 10000 + 5 * t  # e-commerce growth
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 1500  # weekday operations
    elif dow == 5: weekly[i] = -2000  # Saturday reduced
    else: weekly[i] = -4000  # Sunday minimal

# Retail seasonal peaks
seasonal = 1500 * np.sin(2 * np.pi * (t - 250) / 365)  # Q4 peak

# Black Friday + holiday surge
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if m == 11 and 24 <= d <= 30: holiday[i] = 5000  # Black Friday week
    elif m == 12 and 1 <= d <= 20: holiday[i] = 3000  # Christmas rush
    elif m == 7 and d == 15: holiday[i] = 2000  # Prime Day-like

noise = np.random.normal(0, 500, N_POINTS)
units_shipped = base + weekly + seasonal + holiday + noise
units_shipped = np.maximum(units_shipped, 1000).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'units_shipped': units_shipped})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,units_shipped
0,2022-01-01,9625
1,2022-01-02,7302
2,2022-01-03,13189
3,2022-01-04,13620
4,2022-01-05,12735


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('units_shipped Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("units_shipped Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

units_shipped Statistics:
count      730.000000
mean     12286.719178
std       3033.023069
min       4844.000000
25%      10619.250000
50%      12448.000000
75%      14144.500000
max      21937.000000
Name: units_shipped, dtype: float64

CV: 0.247
Skewness: 0.052


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:     3001.9 | RMSE:     3251.9 | MAPE: 19.68%
Seasonal Naive (7)             | MAE:     1731.0 | RMSE:     2317.5 | MAPE: 12.17%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared          RMSE  Time Taken
Model                                                                            
LinearSVR                         897.275337 -67.944257  17637.161776    0.007816
GaussianProcessRegressor          860.076883 -65.082837  17267.282401    0.076761
MLPRegressor                      808.174958 -61.090381  16737.547221    0.284893
KernelRidge                       420.278699 -31.252208  12063.112413    0.021626
DummyRegressor                    119.603825  -8.123371   6415.896103    0.007826
NuSVR                             113.393314  -7.645640   6245.658219    0.019345
QuantileRegressor                 112.024829  -7.540371   6207.518627    0.064673
SVR                               110.996105  -7.461239   6178.693165    0.029394
DecisionTreeRegressor              35.059590  -1.619968   3438.172667    0.016254
AdaBoostRegressor                  16.390448  -0.183881   2311.179137    0.14

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:     1241.5 | RMSE:     1411.6 | MAPE:  6.64%
FLAML Test (catboost)          | MAE:     1524.7 | RMSE:     1752.9 | MAPE: 10.57%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:     2016.8 | RMSE:     2291.7 | MAPE: 14.44%
SF AutoETS                     | MAE:     2645.6 | RMSE:     2976.1 | MAPE: 18.87%
SF SeasonalNaive               | MAE:     2664.9 | RMSE:     2970.7 | MAPE: 19.03%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model         MAE        RMSE      MAPE
     FLAML (catboost) 1241.514482 1411.577655  6.644605
FLAML Test (catboost) 1524.721336 1752.893394 10.566577
   Seasonal Naive (7) 1731.000000 2317.452635 12.170587
         SF AutoARIMA 2016.799072 2291.729369 14.444854
           SF AutoETS 2645.578369 2976.134910 18.869828
     SF SeasonalNaive 2664.928467 2970.708670 19.028069
   Naive (Last Value) 3001.928571 3251.929768 19.675962

Top 3:
                model         MAE        RMSE      MAPE
     FLAML (catboost) 1241.514482 1411.577655  6.644605
FLAML Test (catboost) 1524.721336 1752.893394 10.566577
   Seasonal Naive (7) 1731.000000 2317.452635 12.170587


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -325.73, Std: 1722.36


## Interpretation and Business Insight

- Warehouse throughput follows a strict Mon-Fri operational pattern
- Q4 (Oct-Dec) is the peak season driven by retail holidays
- Black Friday week has the highest throughput of the year
- E-commerce growth creates a strong upward trend
- Sunday operations are minimal (skeleton crew)

## Limitations

1. Synthetic — real throughput depends on inbound inventory, orders, capacity
2. No inbound shipment data
3. No labor availability constraints
4. No product mix information
5. No returns/reverse logistics volume

## How to Improve This Project

1. Add inbound shipment forecasts as leading indicator
2. Include labor availability and shift constraints
3. Model by product category and pick complexity
4. Add returns volume forecasting
5. Use order pipeline data for short-term accuracy

## Production Considerations

- Daily throughput forecasting for labor scheduling
- Integration with WMS (warehouse management system)
- Dock door allocation optimization
- Overtime and temp labor planning

## Common Mistakes

1. Not planning for seasonal peaks early enough
2. Using average throughput for daily scheduling
3. Ignoring capacity constraints (max throughput per shift)
4. Not accounting for inbound delays affecting outbound

## Mini Challenge / Exercises

1. Add a promotional calendar feature and measure throughput lift
2. Model weekday vs weekend operations separately
3. Build a labor requirement forecast from throughput forecast
4. Detect capacity constraint days (throughput plateaus)

## Final Summary / Key Takeaways

1. Warehouse throughput has strong weekly and seasonal patterns
2. Q4 retail holidays drive the annual peak
3. E-commerce growth creates a structural upward trend
4. Labor scheduling is the primary use case for forecasts
5. Inbound inventory visibility is the key missing predictor

In [18]:
import json
metrics = {
    'project': 'Warehouse Throughput Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Warehouse Throughput Forecasting — Complete ===")

Metrics saved.

=== Warehouse Throughput Forecasting — Complete ===
